# Reconhecimento de Cartas de Baralho — Notebook de Treino (Google Colab)

Projeto acadêmico (ICD/ADS) de visão computacional.

**Tarefa:** classificação de cartas — 53 classes (52 cartas + coringa).

**Pipeline deste notebook:**
1. Setup do ambiente (clonar repo + instalar deps + checar GPU)
2. Obter o dataset (Kaggle)
3. Análise exploratória (EDA)
4. Baseline (HOG + Regressão Logística)
5. Modelo principal (transfer learning — EfficientNet-B0)
6. Avaliação no conjunto de teste
7. Experimentos controlados (item 2.4 do enunciado)
8. Avaliação OOD (baralho de design diferente — web)
9. Demo de predição
10. Exportar o modelo

> **Antes de começar:** `Ambiente de execução → Alterar o tipo de hardware → GPU (T4)`.

## 1. Setup do ambiente

In [ ]:
# Edite REPO_URL com o seu repositório no GitHub (passo humano: criar e dar push).
# Se preferir, faça upload manual da pasta src/ para o Colab e pule o clone.
REPO_URL = "https://github.com/nicolasfvp/reconhecimento-cartas.git"
PROJECT_DIR = "reconhecimento-cartas"

import os, sys, subprocess
try:
    import src  # ja estamos dentro do repo (src/ no path)?
    print("Pacote 'src' ja disponivel em:", os.getcwd())
except ModuleNotFoundError:
    if not os.path.isdir(PROJECT_DIR):
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, os.getcwd())
    print("Diretorio do projeto:", os.getcwd())

# No Colab, torch/torchvision/sklearn/matplotlib ja vem prontos.
# Instalamos apenas o que pode faltar.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-image", "kagglehub"], check=True)
print("Setup concluido.")

In [ ]:
import torch
print("Torch:", torch.__version__, "| CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from src.seed import set_seed
set_seed(42)
print("Seed fixada em 42.")

In [ ]:
# (Opcional, recomendado no Colab) Monta o Google Drive e habilita backup de
# checkpoints E figuras, para NAO perder o trabalho se o runtime reiniciar.
# Em Kaggle/local, apenas avisa e segue (baixe manualmente com files.download).
DRIVE_BACKUP_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    DRIVE_BACKUP_DIR = "/content/drive/MyDrive/cartas_backup"
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    print("Drive montado | backup em:", DRIVE_BACKUP_DIR)
except Exception as e:
    print("Drive nao montado (ok no Kaggle/local):", repr(e))

def backup_para_drive(dirs=("models", "reports")):
    """Copia checkpoints e figuras/relatorios para o Drive, se montado.

    Seguro chamar sempre: sem Drive, apenas avisa e nao faz nada.
    """
    import os, shutil
    if not DRIVE_BACKUP_DIR:
        print("Sem Drive -> backup pulado (use files.download para guardar manualmente).")
        return
    for d in dirs:
        if os.path.isdir(d):
            shutil.copytree(d, os.path.join(DRIVE_BACKUP_DIR, d), dirs_exist_ok=True)
    print("Backup atualizado em", DRIVE_BACKUP_DIR)

## 2. Obter o dataset

Usamos o **Cards Image Dataset-Classification** (gpiosenka). O download via `kagglehub` pode pedir login no Kaggle (token `kaggle.json`). Veja `data/README.md` para a alternativa via Kaggle API.

In [ ]:
import kagglehub, os
dataset_path = kagglehub.dataset_download("gpiosenka/cards-image-datasetclassification")
print("Dataset baixado em:", dataset_path)
print("Conteudo:", os.listdir(dataset_path))
DATA_DIR = dataset_path  # deve conter train/ valid/ test/

## 3. Análise exploratória (EDA)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from src.data import _resolve_split_dir

train_dir = _resolve_split_dir(DATA_DIR, "train")
classes = sorted([d.name for d in Path(train_dir).iterdir() if d.is_dir()])
counts = {c: len(list((Path(train_dir) / c).glob("*"))) for c in classes}
print(f"{len(classes)} classes | total de imagens de treino: {sum(counts.values())}")
print("Min/Max imagens por classe:", min(counts.values()), "/", max(counts.values()))

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, c in zip(axes.ravel(), classes[:10]):
    img_path = next((Path(train_dir) / c).glob("*"))
    ax.imshow(Image.open(img_path)); ax.set_title(c, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Baseline (HOG + Regressão Logística)

Linha de base barata (CPU) para comparar com o transfer learning. Usamos um subconjunto por classe para acelerar.

In [ ]:
from src.baseline import run_baseline
baseline_res = run_baseline(DATA_DIR, img_size=128, classifier="logreg", max_per_class=80, seed=42)
print(f"Baseline HOG+LogReg | accuracy={baseline_res['accuracy']:.4f} | "
      f"macro-F1={baseline_res['macro_f1']:.4f} "
      f"(treino={baseline_res['n_train']}, teste={baseline_res['n_test']})")

## 5. Modelo principal (transfer learning - EfficientNet-B0)

Fase 1: backbone congelado (feature extraction). Fase 2: fine-tuning com cosine annealing (LR de pico 3e-4).

In [ ]:
from src.config import Config
from src.train import train_model

# ATENCAO: sem o Google Drive montado, um reset do runtime apaga o modelo (~1h de GPU).
if not globals().get("DRIVE_BACKUP_DIR"):
    print("=" * 64)
    print("ATENCAO: Google Drive NAO montado -> sem backup automatico.")
    print("Se o runtime reiniciar, voce PERDE o modelo e as figuras desta sessao.")
    print("Recomendado: rode a celula de 'Backup no Google Drive' (secao 1) ANTES.")
    print("=" * 64)

cfg = Config(
    data_dir=DATA_DIR,
    backbone="efficientnet_b0",
    epochs_head=8,
    epochs_finetune=20,   # lr_finetune=3e-4 + cosine annealing (defaults do Config)
    augment=True,
    batch_size=32,
    out_dir="models",
    seed=42,
)
result = train_model(cfg)
print("Melhor val_acc:", round(result['best_val_acc'], 4))

# Backup imediato do checkpoint (evita perder o modelo se o runtime reiniciar).
try:
    backup_para_drive()
except NameError:
    print("(rode a celula de montar o Drive para habilitar o backup automatico)")

## 6. Avaliação no conjunto de teste

In [ ]:
from src.data import build_dataloaders
from src.evaluate import evaluate_loader

loaders, class_names = build_dataloaders(DATA_DIR, batch_size=32, augment=False, num_workers=2)
device = cfg.resolve_device()
metrics = evaluate_loader(result["model"], loaders["test"], class_names, device,
                          cm_path="reports/confusion_matrix_test.png")
print(f"TESTE | accuracy={metrics['accuracy']:.4f} | macro-F1={metrics['macro_f1']:.4f}")
print("\nConfusoes mais comuns (verdadeiro -> previsto):")
for t, p, n in metrics["top_confusions"][:10]:
    print(f"  {t} -> {p}: {n}")

try:
    backup_para_drive()  # salva reports/confusion_matrix_test.png no Drive
except NameError:
    pass

In [ ]:
# 6b. Relatorio por classe (precision/recall/F1) -> salva em reports/ para o relatorio.
# Usa metrics["report"] (classification_report) ja calculado pela celula anterior.
import os
import pandas as pd

df_classes = pd.DataFrame(metrics["report"]).transpose().round(3)
os.makedirs("reports", exist_ok=True)
df_classes.to_csv("reports/classification_report_test.csv")

medias = [i for i in ["accuracy", "macro avg", "weighted avg"] if i in df_classes.index]
por_classe = df_classes.drop(index=medias)
print("10 classes com menor F1 no teste (onde o modelo mais erra):")
print(por_classe.sort_values("f1-score").head(10)[["precision", "recall", "f1-score", "support"]].to_string())

try:
    backup_para_drive()  # salva reports/classification_report_test.csv no Drive
except NameError:
    pass

## 7. Experimentos controlados (item 2.4)

Comparamos: (1) feature extraction vs fine-tuning; (2) com vs sem data augmentation. Atencao: roda 3 treinos - leva alguns minutos na GPU.

In [ ]:
import pandas as pd
from src.config import Config
from src.train import train_model
from src.data import build_dataloaders
from src.evaluate import evaluate_loader

# epochs_finetune=20 com lr_finetune=3e-4 + cosine annealing (defaults do Config),
# para o fine-tuning convergir perto do teto (~95%) em vez de parar sub-treinado.
experimentos = {
    "FE (congelado)":          Config(data_dir=DATA_DIR, finetune=False, epochs_head=8, augment=True,  out_dir="models/exp_fe"),
    "FE + fine-tuning":        Config(data_dir=DATA_DIR, finetune=True,  epochs_head=8, epochs_finetune=20, augment=True,  out_dir="models/exp_ft"),
    "FE+FT sem augmentation":  Config(data_dir=DATA_DIR, finetune=True,  epochs_head=8, epochs_finetune=20, augment=False, out_dir="models/exp_noaug"),
}

linhas = []
for nome, c in experimentos.items():
    print(f"\n===== Experimento: {nome} =====")
    r = train_model(c)
    ld, cn = build_dataloaders(DATA_DIR, batch_size=32, augment=False, num_workers=2)
    m = evaluate_loader(r["model"], ld["test"], cn, c.resolve_device())
    linhas.append({"experimento": nome, "val_acc": round(r["best_val_acc"], 4),
                   "test_acc": round(m["accuracy"], 4), "test_macroF1": round(m["macro_f1"], 4)})

df = pd.DataFrame(linhas)
import os; os.makedirs("reports", exist_ok=True)
df.to_csv("reports/experimentos.csv", index=False)
print("\nResumo dos experimentos:")
print(df.to_string(index=False))

# Backup dos 3 checkpoints + experimentos.csv.
try:
    backup_para_drive()
except NameError:
    print("(rode a celula de montar o Drive para habilitar o backup automatico)")

## 8. Avaliação OOD — baralho de *design diferente* (web)

Esta entrega usa um conjunto OOD de **design diferente**: imagens limpas de licença livre,
montadas de forma reprodutível por `src/ood_design.py` (53 classes, mesmos nomes do dataset).

Ele mede o **gap de design** (o modelo aprendeu o conceito da carta ou decorou o estilo do
treino?), **não** o gap de condições de captura (luz/sombra/fundo de fotos reais). Como as
imagens são limpas, o gap medido é um **limite inferior** do esperado em uso real — fotografar
um baralho físico fica como trabalho futuro. Detalhes: `docs/guia_ood_design_web.md`.

In [ ]:
import os
from src.ood_design import build_ood_design_set
from src.evaluate import evaluate_ood

# Conjunto OOD de DESIGN DIFERENTE (imagens limpas da web), gerado de forma reprodutivel.
# Mede o gap de DESIGN, nao o de captura (fotos reais) -> ver docs/guia_ood_design_web.md.
OOD_DIR = "data/raw/ood_design_web"
if not (os.path.isdir(OOD_DIR) and any(os.scandir(OOD_DIR))):
    build_ood_design_set(OOD_DIR)  # baixa os PNGs e monta as 53 pastas de classe

ood = evaluate_ood(result["model"], OOD_DIR, class_names, cfg.resolve_device(),
                   img_size=224, cm_path="reports/confusion_matrix_ood.png")
print(f"OOD (design diferente) | accuracy={ood['accuracy']:.4f} | macro-F1={ood['macro_f1']:.4f} "
      f"| imagens={ood['n_images']} | classes presentes={ood['n_classes_present']}")
print(f"\nGap de DESIGN vs teste Kaggle: {metrics['accuracy'] - ood['accuracy']:+.4f} de accuracy.")
print("Obs.: imagens limpas => este gap e um LIMITE INFERIOR. O gap de captura (fotos reais)")
print("nao e medido aqui e fica como trabalho futuro (docs/guia_coleta_baralho_real.md).")

try:
    backup_para_drive()  # salva reports/confusion_matrix_ood.png no Drive
except NameError:
    pass

In [ ]:
# 8b. Gap de DESIGN: modelo COM aug vs SEM aug (fecha o Experimento 2 do lado da robustez).
# Usa os checkpoints salvos pela secao 7 -> NAO precisa retreinar.
import os
from src.predict import load_model
from src.evaluate import evaluate_ood
from src.ood_design import build_ood_design_set

OOD_DIR = "data/raw/ood_design_web"
if not (os.path.isdir(OOD_DIR) and any(os.scandir(OOD_DIR))):
    build_ood_design_set(OOD_DIR)

for nome, ckpt in [("COM aug", "models/exp_ft/efficientnet_b0_best.pt"),
                   ("SEM aug", "models/exp_noaug/efficientnet_b0_best.pt")]:
    if not os.path.exists(ckpt):
        print(f"[pular] checkpoint ausente: {ckpt} (rode a secao 7 antes).")
        continue
    model_i, class_names_i, cfg_i = load_model(ckpt)
    m = evaluate_ood(model_i, OOD_DIR, class_names_i, cfg_i.resolve_device(), img_size=224)
    print(f"OOD {nome}: acc={m['accuracy']:.4f} | macro-F1={m['macro_f1']:.4f} | imagens={m['n_images']}")

# Interpretacao: se OOD(COM aug) > OOD(SEM aug), a augmentation troca um pouco de
# acuracia in-distribution por robustez. Se nao, e um resultado negativo honesto.
# Backup final (inclui as matrizes de confusao e o experimentos.csv).
try:
    backup_para_drive()
except NameError:
    pass

## 9. Demo de predição em uma imagem

In [ ]:
from pathlib import Path
from src.predict import predict_image
from src.data import _resolve_split_dir

test_dir = _resolve_split_dir(DATA_DIR, "test")
exemplo = next(Path(test_dir).rglob("*.jpg"))
preds = predict_image(result["model"], exemplo, class_names, cfg, topk=3)
print("Imagem:", exemplo)
for i, (nome, prob) in enumerate(preds, 1):
    print(f"  {i}. {nome}: {prob*100:.1f}%")

## 10. Exportar o modelo

O melhor checkpoint já foi salvo em `models/`. Para baixá-lo do Colab:

In [ ]:
# 10. Exportar o modelo. O melhor checkpoint foi salvo em models/ e (se o Drive
# estiver montado) copiado para MyDrive/cartas_backup/models/ pelos backups automaticos.
import glob, os
locais = glob.glob("models/**/*.pt", recursive=True)
print("Checkpoints locais:", locais if locais else "(nenhum no disco)")
print("No Drive (recomendado): MyDrive/cartas_backup/models/")
# Download direto pelo navegador (OPCIONAL; pode TRAVAR se o popup do Colab estiver
# bloqueado -- nesse caso use o Drive). Descomente a linha abaixo para tentar:
# from google.colab import files; files.download("models/efficientnet_b0_best.pt")

In [ ]:
# 10b. Reunir figuras/tabelas de reports/ num zip e enviar ao Drive (confiavel, nao trava).
import os, shutil, glob

artefatos = sorted(glob.glob("reports/*.png") + glob.glob("reports/*.csv"))
print("Artefatos em reports/:", artefatos if artefatos else "(vazio -- rode 6, 6b, 7, 8 antes)")

if artefatos:
    zip_path = shutil.make_archive("reports_figuras", "zip", "reports")
    print("Zip criado:", zip_path, "(", os.path.getsize(zip_path), "bytes )")
    if os.path.isdir("/content/drive/MyDrive"):
        shutil.copy(zip_path, "/content/drive/MyDrive/reports_figuras.zip")
        print(">> Pegue em MyDrive/reports_figuras.zip (nao trava como o files.download)")
    # Download direto pelo navegador (OPCIONAL; descomente se quiser):
    # from google.colab import files; files.download(zip_path)